# predict clinical efficacy on external-1 with fullfeature

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import os
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import shap
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import t as student_t

import matplotlib.pyplot as plt
import itertools
from itertools import combinations
from tqdm import tqdm
from typing import Dict, Any, List, Tuple, Optional

METRIC_NAMES = ["Accuracy", "Precision", "Recall", "F1-score", "AUROC", "AUPRC"]
MODEL_COLORS_ROC = ['#81989B', '#D19246', '#B5AF8B', '#7EA4B6', '#71A682', '#86BC79', '#B8DBB3', '#4A4F7E']
METRIC_COLORS = ['#6AD1A3', '#7FBDDA', '#BBC7BE','#FFD47D', '#FFA288', '#C49892']
MODEL_COLORS_BAR = ['#6AD1A3', '#7FBDDA', '#BBC7BE', '#FFD47D', '#FFA288', '#C49892', '#84ADDC']
    
TRAINSET_PATH='../../datasets/train_set.csv'
EXTERNAL_PATH='../../datasets/external-1.csv'  
SHAP_KERNEL_BG = 60
SHAP_KERNEL_VAL = 120
SHAP_LINEAR_BG = 200
SAVE_DIR = "results"
TARGET = "Clinical_outcome"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# -----------------------------
# feature selection
# -----------------------------
def feature_selection(df, target):

    # Medication features (fixed as input, must be included for non-resistance tasks)
    polyType_cols = ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq']
    Combination_medication_cols = [
        'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose',
        'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose',
        'aminoglycoside_daily_dose'
    ]
    medication_features = polyType_cols + Combination_medication_cols

    time_features = ['Pre_Hospital_Days', 'Pre_ICU_Days']
    base_cols = ['Age', 'Gender', 'BMI']

    comorb_cols = ['Diabetes Mellitus', 'Hypertension', 'Heart Disease', 'Stroke', 'Malignant Tumor',
        'Chronic Kidney Disease', 'Chronic Liver Disease', 'COPD', 'Comorb_other']

    df = df.copy()
    df[comorb_cols] = df[comorb_cols].fillna(0)
    df['Comorb_count'] = df[comorb_cols].sum(axis=1)
    comorb_cols = comorb_cols + ['Comorb_count']

    immuno_cols = ['Use immunosuppressive agents', 'Neutrophil Reduction', 'HIV/AIDS',
        'Post-Transplant Status', 'Chemotherapy/Radiation', 'immuno_Other']

    support_cols = ['Resp_support', 'Oxygen_concentration']

    pre_lab_cols = ['WBC', 'N_percent', 'L_count', 'PLT', 'CRP1', 'PCT1', 'D-d', 'Cr_baseline', 'eGFR1', 'RRT', 'ALT', 'AST', 'TB', 'ALB']
    dynamic_lab_cols = []
    lab_cols = pre_lab_cols + dynamic_lab_cols

    infection_cols = ['Infection_HAP', 'Infection_VAP']
    Coinfection_cols = ['Coinfection_G_Pos', 'Coinfection_G_Neg', 'Coinfection_Fungi']
    df[Coinfection_cols] = df[Coinfection_cols].fillna(0).astype(int)
    infection_cols = infection_cols + Coinfection_cols

    # ===== RSI mapping (R/I=1, S=0) =====
    resistance_features = ['resistance_SXT', 'resistance_KAN', 'resistance_MIN', 'resistance_TGC', 'resistance_CFP-SUL', 'resistance_TOB']
    mapping = {'R': 1, 'I': 1, 'S': 0}

    for c in resistance_features:
        if c in df.columns:
            s = df[c].astype(str).str.strip().str.upper()
            df[c] = pd.to_numeric(s.map(mapping), errors="coerce")

    # Only these two groups are collapsed in SHAP ranking / group enumeration
    group_defs = {
        "Comorbidity": [c for c in comorb_cols if c in df.columns],
        "Immunosuppression": [c for c in immuno_cols if c in df.columns],
    }

    if target == 'Target_Polymyxin':
        feature_cols = base_cols + time_features + comorb_cols + immuno_cols + support_cols + pre_lab_cols
        include_med = False
    else:
        feature_cols = (
            base_cols + comorb_cols + immuno_cols + support_cols +
            lab_cols + infection_cols + resistance_features +
            polyType_cols + Combination_medication_cols
        )
        include_med = True

    # Keep only existing columns (robust to missing cols in some tasks/files)
    feature_cols = [c for c in feature_cols if c in df.columns]

    return feature_cols, df, medication_features, group_defs, include_med

In [3]:
# -----------------------------
# Models
# -----------------------------
def get_models(scale_pos_weight: float):
    models={
        'XGBoost': XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='auc', verbosity=0,  random_state=42, n_jobs=1),
        'LightGBM': LGBMClassifier(class_weight='balanced', verbosity=-1, random_state=42, n_jobs=1),
        'CatBoost': CatBoostClassifier(auto_class_weights='Balanced', verbose=0, allow_writing_files=False, random_state=42, thread_count=1),
        'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1),
        'LogisticRegression': LogisticRegression(class_weight='balanced',random_state=42, max_iter=1000),
        'SVM': SVC(class_weight='balanced',random_state=42, probability=True),
        'KNN': KNeighborsClassifier(weights='distance', n_jobs=1),
    }
    return models

In [4]:
# -----------------------------
# Pipeline utilities
# -----------------------------
TREE_MODELS = {"XGBoost", "LightGBM", "CatBoost", "RandomForest"}
def is_tree(model_name): 
    return model_name in TREE_MODELS


def create_pipeline(model, model_name: str, use_smote: bool = False, smote_ratio: float = 0.8):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if not is_tree(model_name):
        steps.append(("scaler", StandardScaler()))

    if use_smote:
        steps.append(("smote", SMOTE(random_state=42, sampling_strategy=smote_ratio)))

    steps.append(("clf", clone(model)))

    if use_smote:
        return ImbPipeline(steps)
    return Pipeline(steps)


def get_preprocess_transformer(pipe: Pipeline):
    if "clf" not in pipe.named_steps:
        raise ValueError("Pipeline must have a 'clf' step.")
    return pipe[:-1]

In [5]:
# -----------------------------
# CV-SHAP for all models
# -----------------------------
def _sample_indices(n, k, seed):
    if k is None or k <= 0 or k >= n:
        return np.arange(n)
    rng = np.random.RandomState(seed)
    return rng.choice(n, size=k, replace=False)

def _ensure_shap_2d(shap_vals):
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3 and shap_vals.shape[2] == 2:
        shap_vals = shap_vals[:, :, 1]
    return shap_vals


def cv_shap_importance(models, X, y, cv=5, random_state=42,
    medication_features=None, group_defs=None, exclude_medication_in_ranking=True,
    use_smote=False, verbose=True, return_oof_shap=True,
    kernel_background_samples=SHAP_KERNEL_BG, kernel_val_samples=SHAP_KERNEL_VAL, linear_background_samples=SHAP_LINEAR_BG):
    assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
    y = pd.Series(y).astype(int)

    medication_features = medication_features or []
    group_defs = group_defs or {"Comorbidity": [], "Immunosuppression": []}

    # map feature -> group (only two groups)
    feat_to_group = {}
    for g, cols in group_defs.items():
        for c in cols:
            if c in X.columns:
                feat_to_group[c] = g

    kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    results: Dict[str, Any] = {}

    for model_name, model in models.items():
        if verbose:
            print(f"\n===== CV-SHAP (pipeline): {model_name} =====")

        fold_imps = []
        list_shap = []
        list_Xt = []

        for fold, (train, val) in enumerate(kfold.split(X, y), start=1):
            X_train, X_val = X.iloc[train], X.iloc[val]
            y_train = y.iloc[train].values

            pipe = create_pipeline(model, model_name, use_smote=use_smote)
            pipe.fit(X_train, y_train)

            preprocess = get_preprocess_transformer(pipe)
            clf = pipe.named_steps["clf"]

            # Transform to model feature space (numeric array)
            Xt_train = preprocess.transform(X_train)
            Xt_val = preprocess.transform(X_val)

            # choose explainer by model type
            if is_tree(model_name):
                explainer = shap.TreeExplainer(clf)
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            elif model_name == "LogisticRegression":
                # background in transformed space
                idx_bg = _sample_indices(Xt_train.shape[0], linear_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]
                explainer = shap.LinearExplainer(clf, Xt_bg, feature_perturbation="interventional")
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            else:
                # KernelExplainer for SVM/KNN (slow): subsample background and validation
                idx_bg = _sample_indices(Xt_train.shape[0], kernel_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]

                idx_va = _sample_indices(Xt_val.shape[0], kernel_val_samples, seed=random_state + 1000 + fold)
                Xt_va_use = Xt_val[idx_va]

                # function expects transformed features
                def f(z):
                    z = np.asarray(z)
                    if hasattr(clf, "predict_proba"):
                        return clf.predict_proba(z)[:, 1]
                    if hasattr(clf, "decision_function"):
                        s = clf.decision_function(z)
                        return 1.0 / (1.0 + np.exp(-s))
                    raise ValueError(f"{clf.__class__.__name__} lacks predict_proba and decision_function.")

                explainer = shap.KernelExplainer(f, Xt_bg)
                shap_vals = explainer.shap_values(Xt_va_use, nsamples="auto")
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_va_use

            # align with original feature columns (one-hot already, preprocess doesn't change feature count)
            if shap_vals.ndim != 2 or shap_vals.shape[1] != X.shape[1]:
                raise ValueError(
                    f"{model_name} fold {fold}: SHAP shape {shap_vals.shape} != (n, {X.shape[1]}). "
                    "Make sure X is already one-hot numeric and pipeline doesn't change feature count."
                )
            if return_oof_shap:
                list_shap.append(shap_vals)
                list_Xt.append(Xt_used)
            mean_abs = np.abs(shap_vals).mean(axis=0)
            fold_imps.append({feat: float(mean_abs[i]) for i, feat in enumerate(X.columns)})

            if verbose:
                print(f"  fold {fold}: done (val_n={len(X_val)})")

        # average across folds
        avg_imp = {feat: float(np.mean([d.get(feat, 0.0) for d in fold_imps])) for feat in X.columns}
        raw_df = (
            pd.DataFrame({"feature": list(avg_imp.keys()), "mean_abs_shap": list(avg_imp.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        if exclude_medication_in_ranking and medication_features:
            raw_rank_df = raw_df[~raw_df["feature"].isin(medication_features)].reset_index(drop=True)
        else:
            raw_rank_df = raw_df.copy()

        # group aggregation (only two groups collapsed)
        group_scores = {}
        
        for _, row in raw_rank_df.iterrows():
            feat = str(row["feature"])
            val = float(row["mean_abs_shap"])
            g = feat_to_group.get(feat, feat)
            group_scores[g] = group_scores.get(g, 0.0) + val

        grouped_df = (
            pd.DataFrame({"group": list(group_scores.keys()), "mean_abs_shap": list(group_scores.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        results[model_name] = {
            "feature_importance": raw_df,
            "feature_importance_grouped": grouped_df,
        }
        if return_oof_shap and list_shap:
            results[model_name]["shap_oof"] = np.vstack(list_shap)
            results[model_name]["Xt_oof"] = np.vstack(list_Xt)
            results[model_name]["feature_names"] = X.columns.tolist()

    return results

## Evaluate

In [ ]:
# -----------------------------
# Metrics / CV report
# -----------------------------
def evaluate_model(y_true, y_pred, y_proba):
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1-score": float(f1_score(y_true, y_pred, zero_division=0)),
        "AUROC": float(roc_auc_score(y_true, y_proba)),
        "AUPRC": float(average_precision_score(y_true, y_proba)),
    }

## Main

In [ ]:
# ===== 1. load data =====
train_df = pd.read_csv(TRAINSET_PATH, encoding='gbk')
external_df = pd.read_csv(EXTERNAL_PATH, encoding='gbk')

target = TARGET

train_df = train_df[train_df[target].notna()].copy()
train_df[target] = train_df[target].astype(int)
external_df = external_df[external_df[target].notna()].copy()
external_df[target] = external_df[target].astype(int)

print("Train:", train_df[target].value_counts())
print(f"Train shape: {train_df.shape}")
print("\nExternal:", external_df[target].value_counts())
print(f"External shape: {external_df.shape}")

# ===== 2. feature selection =====
feature_cols, train_df, medication_features, group_defs, include_med = feature_selection(train_df, target)
_, external_df, _, _, _ = feature_selection(external_df, target)

X_train = train_df[feature_cols]
y_train = train_df[target]

cols_common = [c for c in feature_cols if c in external_df.columns]
cols_missing = [c for c in feature_cols if c not in external_df.columns]
if cols_missing:
    print(f"\nWarn: external missing columns: {cols_missing}")
X_external = external_df[cols_common].copy()
for c in cols_missing:
    X_external[c] = X_train[c].median()
X_external = X_external[feature_cols]
y_external = external_df[target].values

# ===== 3. models =====
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = get_models(scale_pos_weight)

Train: Clinical_outcome
1    327
0    114
Name: count, dtype: int64
Train shape: (441, 126)

External: Clinical_outcome
1    90
0    21
Name: count, dtype: int64
External shape: (111, 126)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = {name: [] for name in models.keys()}

for model_name, model in models.items():
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        pipe = create_pipeline(model, model_name, use_smote=False)
        pipe.fit(X_tr, y_tr)

        proba = pipe.predict_proba(X_val)[:, 1]
        pred = (proba >= 0.5).astype(int)
        fold_scores.append(evaluate_model(y_val.values, pred, proba))

    cv_metrics[model_name] = fold_scores

rows = []
for model_name, scores in cv_metrics.items():
    for fold_idx, s in enumerate(scores, start=1):
        rows.append({"model": model_name, "fold": fold_idx, **{k: s[k] for k in METRIC_NAMES}})
cv_per_fold_df = pd.DataFrame(rows)
cv_per_fold_df.to_csv(f"{SAVE_DIR}/outcome_AllFeatures_cv_per_fold.csv", index=False)

print("5-fold Cross-Validation results (mean ± std):")
print(f"{'Model':<18} {'Accuracy':>12} {'Precision':>12} {'Recall':>12} {'F1-score':>12} {'AUROC':>12} {'AUPRC':>12}")
print("-" * 95)
for model_name, scores in cv_metrics.items():
    means = {k: np.mean([s[k] for s in scores]) for k in METRIC_NAMES}
    stds = {k: np.std([s[k] for s in scores]) for k in METRIC_NAMES}
    acc_str = f"{means['Accuracy']:.4f}±{stds['Accuracy']:.4f}"
    prec_str = f"{means['Precision']:.4f}±{stds['Precision']:.4f}"
    rec_str = f"{means['Recall']:.4f}±{stds['Recall']:.4f}"
    f1_str = f"{means['F1-score']:.4f}±{stds['F1-score']:.4f}"
    auroc_str = f"{means['AUROC']:.4f}±{stds['AUROC']:.4f}"
    auprc_str = f"{means['AUPRC']:.4f}±{stds['AUPRC']:.4f}"
    print(f"{model_name:<18} {acc_str:>12} {prec_str:>12} {rec_str:>12} {f1_str:>12} {auroc_str:>12} {auprc_str:>12}")

5-fold Cross-Validation results (mean ± std):
Model                  Accuracy    Precision       Recall     F1-score        AUROC        AUPRC
-----------------------------------------------------------------------------------------------
XGBoost            0.7235±0.0502 0.7905±0.0290 0.8534±0.0537 0.8202±0.0355 0.6827±0.0357 0.8560±0.0210
LightGBM           0.7166±0.0288 0.7748±0.0144 0.8717±0.0434 0.8198±0.0209 0.6900±0.0453 0.8636±0.0276
CatBoost           0.7483±0.0200 0.7965±0.0247 0.8899±0.0310 0.8398±0.0111 0.6864±0.0242 0.8682±0.0116
RandomForest       0.7596±0.0087 0.7576±0.0075 0.9939±0.0075 0.8598±0.0053 0.6824±0.0316 0.8611±0.0210
LogisticRegression 0.6463±0.0220 0.8112±0.0117 0.6822±0.0358 0.7405±0.0208 0.6306±0.0288 0.8221±0.0225
SVM                0.7528±0.0231 0.7514±0.0179 0.9969±0.0062 0.8568±0.0127 0.6385±0.0283 0.8198±0.0255
KNN                0.7142±0.0366 0.7521±0.0223 0.9172±0.0398 0.8261±0.0241 0.5663±0.0395 0.7832±0.0276


In [ ]:
print("\n" + "=" * 75)
print("External validation (external-1) results:")
print(f"{'Model':<18} {'Accuracy':>8} {'Precision':>10} {'Recall':>8} {'F1-score':>9} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 75)
external_metrics = {}
for model_name, model in models.items():
    pipe = create_pipeline(model, model_name, use_smote=False)
    pipe.fit(X_train, y_train)

    proba = pipe.predict_proba(X_external)[:, 1]
    pred = (proba >= 0.5).astype(int)

    external_metrics[model_name] = evaluate_model(y_external, pred, proba)
    m = external_metrics[model_name]
    print(f"{model_name:<18} {m['Accuracy']:>8.4f} {m['Precision']:>10.4f} {m['Recall']:>8.4f} {m['F1-score']:>9.4f} {m['AUROC']:>8.4f} {m['AUPRC']:>8.4f}")


External validation (external-1) results:
Model              Accuracy  Precision   Recall  F1-score    AUROC    AUPRC
---------------------------------------------------------------------------
XGBoost              0.7568     0.8387   0.8667    0.8525   0.6720   0.8896
LightGBM             0.7748     0.8218   0.9222    0.8691   0.6619   0.8933
CatBoost             0.7568     0.8247   0.8889    0.8556   0.6339   0.8791
RandomForest         0.8018     0.8091   0.9889    0.8900   0.6698   0.8818
LogisticRegression   0.6847     0.8571   0.7333    0.7904   0.6392   0.8702
SVM                  0.8198     0.8182   1.0000    0.9000   0.6534   0.8678
KNN                  0.8018     0.8208   0.9667    0.8878   0.5849   0.8445
